# Heart Disease Diagnosis Prediction
Analyzes patient demographics and clinical attributes to predict heart disease risk using structured clinical data.


In [ ]:
!pip install -q pandas numpy scikit-learn seaborn matplotlib
!curl -L -o heart-disease-data.zip https://www.kaggle.com/api/v1/datasets/download/redwankarimsony/heart-disease-data
!unzip -q -o heart-disease-data.zip -d data/


In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import roc_auc_score, confusion_matrix, classification_report


### 1. Data Cleaning & Outlier Management
Replaces missing values via median/mode imputation and scales numerical features (e.g., chol, trestbps).


In [ ]:
import glob
file_path = glob.glob('data/*.csv')[0]
df = pd.read_csv(file_path)

# Drop duplicates and handle missing values
df = df.drop_duplicates()
df = df.fillna(df.median(numeric_only=True))

# Separate features and target (commonly 'num' or 'target')
target_col = 'num' if 'num' in df.columns else df.columns[-1]
X = df.drop(columns=[target_col, 'id'], errors='ignore')
y = (df[target_col] > 0).astype(int) # Binary classification: 0 = No disease, 1 = Disease

# Scaling
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)


### 2. Model Training


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, stratify=y, random_state=42)

model = GradientBoostingClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)


### 3. Evaluation & Clinical Metrics
Focuses on Recall (Sensitivity) to minimize false negatives in medical diagnostics.


In [ ]:
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print("ROC-AUC Score:", roc_auc_score(y_test, y_prob))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

plt.figure(figsize=(6,4))
sns.heatmap(confusion_matrix(y_test, y_pred), annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.show()
